## Architecture reference for this lab

**Step 13 — Escalation Agent (DynamoDB + Step Functions)**

![Step 13 — Escalation Agent (DynamoDB + Step Functions)](images/step-13-escalation.png)



# Lab 4 — Escalation Agent (DynamoDB + Step Functions)

**What this lab is.** We build the "hand-off to a human" system. When a patient asks
something clinical, CareConnect must not answer — it must create a durable **ticket** and
kick off an **approval workflow** so a licensed clinician can take over. We use a DynamoDB
table (to store tickets) and AWS Step Functions (to run the approval workflow).

**Why we do it.** Refusing a clinical question isn't enough on its own — the patient still
needs help. Escalation makes sure their question reaches a real clinician instead of being a
dead end. And storing it durably means nothing gets lost.

**Why it's needed here.** This is the safety valve of the whole system. Every clinical or
high-risk request ends here, with a tracked ticket and a human in charge.

**How it helps the project.** The Supervisor (lab-05) calls this whenever the safety checks
say "escalate". It closes the loop: refuse to answer *and* route to a human.

**The use case.** "Should I stop my medication before my procedure?" → CareConnect creates a
ticket, starts the approval workflow, and tells the patient a clinician will review it.

---

## Prerequisite — run these two setup cells first (every lab has them)

Before this lab's own steps, run the **two setup cells** below. Every notebook (lab-00
through lab-08) starts with these same two cells — on purpose, not by mistake.

**Why they repeat in every notebook.** Each notebook runs in its own fresh "kernel"
(a separate Python session) with no memory of the other notebooks. So each one has to set
itself up from scratch. These two cells are that setup.

**Cell 1 — "bootstrap":** finds the project's main folder (the one containing `lab_helpers`)
no matter where the notebook is opened from, so `import lab_helpers...` always works.

**Cell 2 — "preflight":** checks all helper files are present before the lab begins, and stops
with a clear message if anything is missing — instead of failing confusingly later.

**Do I run them?** Yes — run both, in order, at the top of **every** lab. They take a second
and prevent the most common setup problems. After these two, continue with the lab's steps.

In [1]:
# WHAT THIS CELL DOES (plain English):
# - It looks at the current folder, then its parent, then its parent's parent, and so on,
#   until it finds the folder that contains "lab_helpers". That folder is our project root.
# - It then switches into that folder and adds it to Python's search path, so that
#   'import lab_helpers...' works from anywhere.
# - If it never finds "lab_helpers", it stops with a clear message instead of a confusing error later.
# You do not need to edit anything here — just run it first.
# === CareConnect bootstrap — run me first ===
# Makes this notebook work from any folder and gives a clear error if the
# lab_helpers package is missing (e.g. not uploaded to the repo).
import os, sys

def _find_repo_root(start=None):
    here = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(here, "lab_helpers")):
            return here
        parent = os.path.dirname(here)
        if parent == here:
            return None
        here = parent

_root = _find_repo_root()
if _root is None:
    raise RuntimeError(
        "Could not find the 'lab_helpers/' folder from " + os.getcwd() + ".\n"
        "This means the helper package is not next to the notebooks.\n"
        "Fix: make sure lab_helpers/ and requirements.txt are in the same folder\n"
        "as these .ipynb files (see README > Setup). Then re-run this cell.")
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print("Repo root:", _root)

Repo root: /home/sagemaker-user/careconnect-patient-assistant-k21


In [2]:
# WHAT THIS CELL DOES (plain English):
# - It checks that each helper file we rely on is actually present on disk.
# - If any are missing, it stops now with a clear list of what to upload, rather than
#   failing in the middle of the lab.
# Run this straight after the bootstrap cell above.
# === Preflight: confirm every helper file is present BEFORE running the lab ===
import os
_required = [
    "requirements.txt",
    "lab_helpers/__init__.py",
    "lab_helpers/utils.py",
    "lab_helpers/careconnect_agents.py",
    "lab_helpers/deterministic_safety.py",
    "lab_helpers/runtime_entrypoint.py",
]
_missing = [f for f in _required if not os.path.isfile(f)]
if _missing:
    raise RuntimeError("Missing required files:\n  - " + "\n  - ".join(_missing) +
        "\n\nUpload the full lab_helpers/ folder + requirements.txt, then re-run.")
print("Preflight OK — all helper files present.")

Preflight OK — all helper files present.


### Step 1 — Create the tickets table (DynamoDB)

**What:** create a DynamoDB table to hold escalation tickets, with automatic expiry (TTL) so
old tickets clean themselves up.

**Why:** we need somewhere durable to record each escalated request. DynamoDB is a simple,
pay-per-use database that's ideal for this. TTL keeps it tidy without manual cleanup.

In [3]:
# WHAT THIS CELL DOES (plain English):
# - Creates a database table called (…-escalations-sdk) to store escalation tickets.
# - 'PAY_PER_REQUEST' means we only pay per use — no fixed cost.
# - Then it turns on TTL (time-to-live): tickets with an 'expires_at' time auto-delete later.
import boto3, time, json
import lab_helpers.utils as u
ddb = boto3.client("dynamodb", region_name=u.REGION)

try:
    ddb.create_table(
        TableName=u.ESCALATION_TABLE,
        AttributeDefinitions=[{"AttributeName":"ticket_id","AttributeType":"S"}],
        KeySchema=[{"AttributeName":"ticket_id","KeyType":"HASH"}],
        BillingMode="PAY_PER_REQUEST")
    ddb.get_waiter("table_exists").wait(TableName=u.ESCALATION_TABLE)
    print("Created table", u.ESCALATION_TABLE)
except ddb.exceptions.ResourceInUseException:
    print("Table already exists")

ddb.update_time_to_live(
    TableName=u.ESCALATION_TABLE,
    TimeToLiveSpecification={"Enabled":True,"AttributeName":"expires_at"})
print("TTL enabled on expires_at")

Created table careconnect-escalations-sdk
TTL enabled on expires_at


### Step 2 — Create the approval workflow (Step Functions)

**What:** create a Step Functions **state machine** — a visual workflow — that represents the
human-approval process. (In this lab it's a simplified version.)

**Why:** approvals are a *process*, not a single action. Step Functions is AWS's tool for
running multi-step processes reliably. In production this workflow would notify a clinician
and wait for their decision.

In [4]:
# WHAT THIS CELL DOES (plain English):
# - Defines a simple approval workflow (wait, then mark approved) and creates it as a Step
#   Functions "state machine".
# - Creates the permission role the workflow needs and saves the workflow's ID for later.
# - In a real system this workflow would alert a clinician and pause for their real decision.
sfn = boto3.client("stepfunctions", region_name=u.REGION)
account = u.get_aws_account_id()

sfn_role_arn = u._create_role(
    u.name("CareConnectSfnRole"), "states.amazonaws.com",
    {"Version":"2012-10-17","Statement":[{"Effect":"Allow","Action":["cloudwatch:*"],"Resource":"*"}]},
    u.name("CareConnectSfnPolicy"))

definition = json.dumps({
  "Comment":"CareConnect Human Approval Workflow (SDK build)",
  "StartAt":"CreateApprovalTask",
  "States":{
    "CreateApprovalTask":{"Type":"Wait","Seconds":60,"Next":"ApprovalCompleted"},
    "ApprovalCompleted":{"Type":"Pass","Result":{"status":"approved"},"End":True}}})

try:
    sm = sfn.create_state_machine(
        name=u.STATE_MACHINE_NAME, definition=definition,
        roleArn=sfn_role_arn, type="STANDARD")
    arn = sm["stateMachineArn"]
    print("Created state machine:", arn)
except sfn.exceptions.StateMachineAlreadyExists:
    arn = f"arn:aws:states:{u.REGION}:{account}:stateMachine:{u.STATE_MACHINE_NAME}"
    print("State machine exists:", arn)
u.put_ssm_parameter(f"{u.SSM_PREFIX}/state_machine_arn", arn)

Created role CareConnectSfnRole-sdk
Created state machine: arn:aws:states:us-east-1:831963379350:stateMachine:careconnect-human-approval-workflow-sdk


### Step 3 — Escalate a real example

**What:** create a ticket for a clinical question and start the approval workflow.

**Why:** this demonstrates the end-to-end hand-off: a ticket is stored, the workflow starts,
and the patient gets a clear "a human will review this" message with a ticket ID.

In [5]:
# WHAT THIS CELL DOES (plain English):
# - Defines 'escalate': it saves a ticket to the database (with a unique ID and a 30-day
#   expiry) and starts the approval workflow.
# - Then it escalates a sample clinical question and prints the ticket details, showing the
#   patient-facing "escalated to a human reviewer" message.
import uuid
from datetime import datetime, timezone
table = boto3.resource("dynamodb", region_name=u.REGION).Table(u.ESCALATION_TABLE)

def escalate(reason, category, request_details):
    ticket_id = str(uuid.uuid4())
    table.put_item(Item={
        "ticket_id": ticket_id, "category": category, "reason": reason,
        "request_details": request_details, "status": "pending_review",
        "created_at": datetime.now(timezone.utc).isoformat(),
        "expires_at": int(datetime.now(timezone.utc).timestamp()) + 2592000})
    ex = sfn.start_execution(stateMachineArn=arn,
                             input=json.dumps({"ticket_id": ticket_id}))
    return {"message":"Your request has been escalated to a human reviewer.",
            "ticket_id": ticket_id, "workflow_execution": ex["executionArn"],
            "status":"pending_review"}

print(escalate("Medication timing requires clinician review.", "clinical_question",
               {"patient_request":"Should I stop my medication before my procedure?"}))

{'message': 'Your request has been escalated to a human reviewer.', 'ticket_id': 'c47f8193-9994-44b0-8718-95110d95a446', 'workflow_execution': 'arn:aws:states:us-east-1:831963379350:execution:careconnect-human-approval-workflow-sdk:4799a26e-4fcf-42fe-9623-5c5cf8cc14f5', 'status': 'pending_review'}


## Lab 4 complete ✅

Durable escalation ticket + approval workflow, all `-sdk`.